# **Statistical Model Comparison**

**Prerequisites:** Cross-Validation, Classification Metrics | **Next:** (end of `05_Model_Evaluation/`) | **Depth tier:** Core

## 1. Theory

`05_Model_Evaluation\02_classification_metrics.ipynb` already covers the paired t-test for
comparing two models' CV scores. This notebook adds the two gaps: a test
appropriate for comparing two classifiers on a **single** test set
(McNemar's - no CV needed), and the correction required when comparing
**more than two** models at once.

## 2. Mathematical Derivation

**McNemar's test** - compares two classifiers on the SAME test set by
looking only at the examples where they *disagree*:
$$\chi^2 = \frac{(|n_{01}-n_{10}|-1)^2}{n_{01}+n_{10}}$$
where $n_{01}$ = count of examples Model A got wrong but Model B got
right, $n_{10}$ = the reverse. Examples both models got right or both got
wrong are **uninformative for comparison** and correctly excluded - the
test asks specifically "when they disagree, does one win more often than
chance (50/50) would predict?" Compare $\chi^2$ against a
$\chi^2$-distribution with 1 degree of freedom.

**Multiple-comparisons correction** - when comparing $k>2$ models
pairwise, running $\binom{k}{2}$ separate tests at $\alpha=0.05$ each
inflates the overall false-positive rate. **Bonferroni correction**
(simplest, conservative): use $\alpha/\binom k2$ as the per-test
significance threshold instead of $\alpha$ directly.

### Worked derivation of why the correction is needed

If you run $m$ independent tests, each with a 5% chance of a false
positive under $H_0$, the probability of **at least one** false positive
among all $m$ is $1-(0.95)^m$, not $5\%$. For $m=10$ tests:
$1-0.95^{10}=1-0.599=0.401$ - a 40% chance of at least one spurious
"significant" result purely from running many comparisons, even if no
real difference exists anywhere.

## 3. Worked Numerical Example

**McNemar's**: Model A and B tested on 200 samples. $n_{01}=18$ (A wrong,
B right), $n_{10}=6$ (A right, B wrong).
$$\chi^2 = \frac{(|18-6|-1)^2}{18+6} = \frac{11^2}{24} = \frac{121}{24} \approx 5.04$$
Critical value for $\chi^2_1$ at $\alpha=0.05$ is 3.84; $5.04>3.84$ →
**reject $H_0$**: Model B is significantly better, specifically on the
disagreement cases.

**Multiple comparisons**: comparing 5 models pairwise gives
$\binom52=10$ tests; Bonferroni-corrected threshold
$=0.05/10=0.005$ - a p-value of, say, 0.03 from one of these pairwise
tests would appear "significant" uncorrected but is NOT significant after
correction (0.03 > 0.005).

In [2]:
## verify both worked examples

import numpy as np
from scipy.stats import chi2

n01, n10 = 18, 6
chi_sq = (abs(n01-n10)-1)**2 / (n01+n10)
p_value = 1 - chi2.cdf(chi_sq, df=1)
print(chi_sq, p_value)   # 5.042, 0.0247 -- significant at 0.05

# statsmodels has this built in directly
from statsmodels.stats.contingency_tables import mcnemar
table = [[150, 18], [6, 26]]   # [both-correct, A-wrong-B-right], [A-right-B-wrong, both-wrong]
result = mcnemar(table, exact=False, correction=True)
print(result.statistic, result.pvalue)

# multiple comparisons inflation
m = 10
print(1 - 0.95**m)   # 0.401 -- matches hand calc

5.041666666666667 0.024744672046398963
5.041666666666667 0.02474467204639891
0.4012630607616213


## 5. Practical Implementation
`statsmodels.stats.multitest.multipletests` implements Bonferroni and
several less-conservative alternatives (Holm, Benjamini-Hochberg) directly
- worth using the library rather than hand-computing corrected thresholds
in practice, once the underlying reasoning above is understood.

## 6. Failure Cases
Cherry-picking the single "most significant" pairwise comparison out of
many run, without correction or without disclosing how many comparisons
were actually made - a genuine and common research-methodology failure
mode, not just a theoretical concern.

## 7. Assumptions
McNemar's test assumes the test set is a single, fixed evaluation set
(not aggregated across CV folds - for that case, the paired t-test from
`05_Model_Evaluation\02_classification_metrics.ipynb` is the appropriate tool instead); both
tests assume the disagreement counts are large enough for the
chi-squared/t approximations to be reasonable (a common rule of thumb:
$n_{01}+n_{10}\ge25$ for McNemar's without a continuity correction).